# Construção da Camada Bronze

Nesse notebook persistiremos os dados do volume em tabelas no schema bronze, sem alteração ou tratamento dos dados.  
Em todas as cargas, usaremos a opção de `InferSchema` como `False`, salvando todos os dados como tipo `string`. Isso garante que nenhuma transformação ocorra nessa etapa.

In [0]:
%sql
USE CATALOG mvp_engenharia_dados;
USE SCHEMA bronze;

In [0]:
%sql
DROP TABLE if exists fato_pontualidade;

## Tabela de voos (VRA)
Leitura dos dados em arquivos no Volume, e persistencia desses dados em uma tabela no schema bronze.

In [0]:
df_pontualidade = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", False)
    .option("sep", ";")
    .load("/Volumes/mvp_engenharia_dados/staging/pontualidade/vra/")
)

print(f"Total de Linhas: {df_pontualidade.count():,d}")
display(df_pontualidade.dtypes)
display(df_pontualidade.limit(10))

Precisamos usar as opcoes `minReaderVersion` e `minWriterVersion` por questão de compatibilidade dos nomes das colunas com acentos e caracteres especiais.  
Esse passo se faz necessário para mantermos os dados como vem da fonte original, sem alterações

In [0]:
(
    df_pontualidade.write.mode("overwrite")
    .option("delta.columnMapping.mode", "name")
    .option("delta.minReaderVersion", "2")
    .option("delta.minWriterVersion", "5")
    .format("delta")
    .saveAsTable("bronze_vra")
)

In [0]:
%sql
select count(*) from bronze_vra

In [0]:
%sql
select * from bronze_vra limit 10

### Documentacao da VRA

In [0]:
%sql
COMMENT ON TABLE bronze_vra IS 'Tabela da camada bronze com os dados brutos de VRA (Voo Regular Ativo).

O VRA é uma base de dados composta por informações de voos de empresas de transporte aéreo regular que apresenta alterações de voos (atrasos, antecipações e cancelamentos), horários em que os voos ocorreram e as justificativas apresentadas pelas empresas aéreas para tais alterações.

Por meio desta base de dados, podem ser obtidos os índices de pontualidade, regularidade e de desempenho operacional, além dos percentuais de atrasos e cancelamentos.

O mês publicado se refere às etapas cujas decolagens eram previstas para o mês em questão ou cujas decolagens, em caso de etapa não prevista, foram realizadas no mês em questão.';

In [0]:
%sql
COMMENT ON COLUMN bronze_vra.`Sigla ICAO Empresa Aérea` IS 'Sigla/Designador ICAO da empresa aérea';
COMMENT ON COLUMN bronze_vra.`Empresa Aérea` IS 'Nome da empresa aérea';
COMMENT ON COLUMN bronze_vra.`Número Voo` IS 'Numeração do voo';
COMMENT ON COLUMN bronze_vra.`Código DI` IS 'Caractere usado para identificar o Dígito Identificador (DI) para cada etapa de voo';
COMMENT ON COLUMN bronze_vra.`Código Tipo Linha` IS 'Caractere usado para identificar o Tipo de Linha realizada para cada etapa de voo';
COMMENT ON COLUMN bronze_vra.`Modelo Equipamento` IS 'Modelo do equipamento utilizado no voo no padrão ICAO';
COMMENT ON COLUMN bronze_vra.`Número de Assentos` IS 'Quantidade de assentos da aeronave';
COMMENT ON COLUMN bronze_vra.`Sigla ICAO Aeroporto Origem` IS 'Sigla/Designador ICAO do aeroporto de origem';
COMMENT ON COLUMN bronze_vra.`Descrição Aeroporto Origem` IS 'Nome do aeroporto, cidade, estado e país no qual é localizado';
COMMENT ON COLUMN bronze_vra.`Partida Prevista` IS 'Data e horário da partida prevista informada pela empresa aérea, em horário de Brasília';
COMMENT ON COLUMN bronze_vra.`Partida Real` IS 'Data e horário da partida realizada informada pela empresa aérea, em horário de Brasília';
COMMENT ON COLUMN bronze_vra.`Sigla ICAO Aeroporto Destino` IS 'Sigla/Designador ICAO do aeroporto de destino';
COMMENT ON COLUMN bronze_vra.`Descrição Aeroporto Destino` IS 'Nome do aeroporto, cidade, estado e país no qual é localizado';
COMMENT ON COLUMN bronze_vra.`Chegada Prevista` IS 'Data e horário da chegada prevista informada pela empresa aérea, em horário de Brasília';
COMMENT ON COLUMN bronze_vra.`Chegada Real` IS 'Data e horário da chegada realizada, informada pela empresa aérea, em horário de Brasília';
COMMENT ON COLUMN bronze_vra.`Situação Voo` IS 'Situação do voo: realizado, cancelado ou não informado';
COMMENT ON COLUMN bronze_vra.`Justificativa` IS 'Justificativa da empresa aérea para alteração do voo. Deixou de ser exigida a partir de abril de 2020, com a revogação da IAC 1504';
COMMENT ON COLUMN bronze_vra.`Referência` IS 'Mês de referência da publicação: período ao qual as decolagens previstas ou realizadas pertencem';
COMMENT ON COLUMN bronze_vra.`Situação Partida` IS 'Situação do voo na partida: Antecipado, Pontual, Atraso 30-60, Atraso 60-120, Atraso 120-240, Atraso > 240';
COMMENT ON COLUMN bronze_vra.`Situação Chegada` IS 'Situação do voo na chegada: Antecipado, Pontual, Atraso 30-60, Atraso 60-120, Atraso 120-240, Atraso > 240';
COMMENT ON COLUMN bronze_vra.`Codeshare` IS 'Indica se o voo é compartilhado entre empresas aéreas (codeshare)';

## Tabelas de Dimensão
Persistencia das tabelas de dimensao no schema bronze
- Aerodromos
- Empresas Aereas

### Aerodromos

In [0]:
df_aerodromos = (
    spark.read
        .format("csv")
        .option("header", True)
        .option("skipRows", 1)
        .option("inferSchema", False)
        .option("encoding", "Windows-1252")
        .option("sep", ";")
        .load("/Volumes/mvp_engenharia_dados/staging/pontualidade/AerodromosPublicos.csv")
)

print(f"Total de Linhas: {df_aerodromos.count():,d}")
display(df_aerodromos.dtypes)
display(df_aerodromos.limit(10))

In [0]:
(
    df_aerodromos.write.mode("overwrite")
    .option("delta.columnMapping.mode", "name")
    .option("delta.minReaderVersion", "2")
    .option("delta.minWriterVersion", "5")
    .format("delta")
    .saveAsTable("bronze_aerodromos")
)

In [0]:
%sql
select COUNT(*) from bronze_aerodromos

In [0]:
%sql
select * from bronze_aerodromos LIMIT 5

### Documentacao Aerodromos

In [0]:
%sql
COMMENT ON TABLE bronze_aerodromos IS 'Tabela da camada bronze com os dados brutos de aeródromos publicos.

Dados cadastrais de aeródromos publicos, que são aqueles aeródromos abertos ao tráfego aéreo de uso público.
O Cadastro de Aeródromos é a informação oficial sobre a infraestrutura de aeródromos civis públicos e privados do Brasil.
O cadastro de aeródromos civis é mantido pela ANAC para inscrição dos aeródromos, instalações e equipamentos de auxílio à navegação aérea para atender à aviação civil.
De acordo com o art. 30 do Código Brasileiro de Aeronáutica (Lei no 7.565, de 19 de dezembro de 1986), nenhum aeródromo civil poderá ser utilizado sem estar devidamente cadastrado.

Algumas definições importantes:

Aeródromo é toda área destinada a pouso, decolagem e movimentação de aeronaves. Os aeródromos podem sem classificados como públicos e privados.

Helipontos são os aeródromos destinados exclusivamente a helicópteros.

Heliportos são os helipontos públicos dotados de instalações e facilidades para apoio de operações a helicópteros e de embarque e desembarque de pessoas e cargas.

Helideque: é uma estrutura construída para pousos e decolagens de helicópteros, instalada a bordo de plataformas marítimas ou em embarcações.';


In [0]:
%sql
 COMMENT ON COLUMN bronze_aerodromos.`Código OACI` IS 'Código de 4 letras utilizado internacionalamente para identificação do aeródromo, conforme regras da OACI. No Brasil, os códigos iniciam com SB, SD, SI, SJ, SN, SS e SW';
COMMENT ON COLUMN bronze_aerodromos.`CIAD` IS 'Código de Identificação do Aeródromo: identificador único de aeródromos civis, formado por dois caracteres da UF e quatro dígitos sequenciais (formato UF0000)';
COMMENT ON COLUMN bronze_aerodromos.`Nome` IS 'Denominação do aeródromo';
COMMENT ON COLUMN bronze_aerodromos.`Município` IS 'Município onde se localiza o aeródromo, considerando as coordenadas de referência';
COMMENT ON COLUMN bronze_aerodromos.`UF` IS 'Unidade da federação do município onde se localiza o aeródromo';
COMMENT ON COLUMN bronze_aerodromos.`Município Servido` IS 'Município principal servido pelo aeródromo';
COMMENT ON COLUMN bronze_aerodromos.`UF Servido` IS 'Unidade da federação do município servido';
COMMENT ON COLUMN bronze_aerodromos.`LATGEOPOINT` IS 'Latitude do ponto de referência do aeródromo em formato decimal. Campo não mencionado na documentação oficial, presente no arquivo CSV';
COMMENT ON COLUMN bronze_aerodromos.`LONGEOPOINT` IS 'Longitude do ponto de referência do aeródromo em formato decimal. Campo não mencionado na documentação oficial, presente no arquivo CSV';
COMMENT ON COLUMN bronze_aerodromos.`Latitude` IS 'Latitude do ponto de referência do aeródromo em graus, minutos e segundos (sistema de referência WGS-84)';
COMMENT ON COLUMN bronze_aerodromos.`Longitude` IS 'Longitude do ponto de referência do aeródromo em graus, minutos e segundos (sistema de referência WGS-84)';
COMMENT ON COLUMN bronze_aerodromos.`Altitude` IS 'Altitude do ponto de referência mais elevado na área de pouso, em metros';
COMMENT ON COLUMN bronze_aerodromos.`Operação Diurna` IS 'Tipo de operação do período diurno para o qual o aeródromo está habilitado (VFR – Visual, IFR – Instrumento)';
COMMENT ON COLUMN bronze_aerodromos.`Operação Noturna` IS 'Tipo de operação do período noturno para o qual o aeródromo está habilitado (VFR – Visual, IFR – Instrumento)';
COMMENT ON COLUMN bronze_aerodromos.`Situação` IS 'Indicação de eventual medida administrativa aplicada pela ANAC ao aeródromo, constituindo-se em algum tipo de restrição operacional';
COMMENT ON COLUMN bronze_aerodromos.`Validade do Registro` IS 'Data de validade do cadastro do aeródromo, renovável mediante requerimento prévio';
COMMENT ON COLUMN bronze_aerodromos.`Portaria de Registro` IS 'Número e ano da última Portaria de cadastro do aeródromo dentro da validade (formato PAxxxx-yyyy)';
COMMENT ON COLUMN bronze_aerodromos.`Link Portaria` IS 'Link para acesso direto ao arquivo .pdf da portaria de cadastro do aeródromo';

### Empresas

In [0]:
df_empresas = (
    spark.read.format("json")
        .option("inferSchema", False)
        .load("/Volumes/mvp_engenharia_dados/staging/pontualidade/pda_empresas_aereas_nacionais.json")
)

print(f"Total de Linhas: {df_empresas.count():,d}")
display(df_empresas.dtypes)
display(df_empresas.limit(10))

In [0]:
(
    df_empresas.write.mode("overwrite")
    .option("delta.columnMapping.mode", "name")
    .option("delta.minReaderVersion", "2")
    .option("delta.minWriterVersion", "5")
    .format("delta")
    .saveAsTable("bronze_empresas")
)

In [0]:
%sql
select COUNT(*) from bronze_empresas

In [0]:
%sql
select * from bronze_empresas LIMIT 5

### Documentacao Empresas

In [0]:
%sql
COMMENT ON TABLE bronze_empresas IS 'Tabela da camada bronze com os dados brutos de empresas habilitadas no Brasil.

Informações do Cadastro de empresas aéreas aptas a operar no Brasil.

Os serviços aéreos públicos abrangem os serviços aéreos públicos especializados e os serviços de transporte aéreo público de passageiro, carga ou mala postal, regular ou não-regular, doméstico ou internacional, conforme definição legal prevista pelo Artigo 175 da Lei no 7.565/86 – Código Brasileiro de Aeronáutica - CBA.

Para que uma empresa brasileira possa explorar serviços aéreos públicos, esta dependerá sempre de prévia concessão, quando se tratar de transporte aéreo regular, ou de autorização no caso de transporte aéreo não-regular ou de serviços aéreos especializados.

A concessão para operar terá validade de até 10 (dez) anos, tornando-se efetiva após a publicação do extrato do contrato celebrado com a ANAC, podendo ser renovada em função do cumprimento do objetivo social e das demais leis e normas infralegais aplicáveis.

A autorização para operar terá validade de até 5 (cinco) anos, contados a partir da data da publicação do ato de outorga, podendo ser renovada, no todo ou em parte, em função do cumprimento do objetivo social relacionado às atividades aéreas e das demais leis e normas infralegais aplicáveis.

Já para as empresas constituídas em outros países, para funcionamento e operação no Brasil, estas deverão inicialmente serem designadas pelo Governo do país de origem pelas vias diplomáticas, após o que deverão solicitar uma Autorização para Funcionamento e, posteriormente, uma Autorização para Operar junto à ANAC, conforme Artigo 205 do Código Brasileiro de Aeronáutica.

As autorizações para funcionamento no Brasil e para operar de empresas estrangeiras não possuem validades definidas, vigorando enquanto as empresas mantiverem sua designação pelo Governo do respectivo país de origem, bem como suas condições técnico-operacionais.

Atualmente, o processo de outorga de serviços aéreos públicos é regido pelas seguintes normas:

§ Lei no 7.565, de 19.12.1986 – Dispõe sobre o Código Brasileiro de Aeronáutica;

§ Lei no 11.182, de 27.09.2005 - Cria a Agência Nacional de Aviação Civil – ANAC, e dá outras providências;

§ Resolução no 377, de 15.03.2016 - Regulamenta a outorga de serviços aéreos públicos para empresas brasileiras e dá outras providências; e

§ Portaria no 616/SAS, de 16.03.2016 - Regulamenta os artigos 7o e 16 da Resolução no 377, de 15.03.2016.';

In [0]:
%sql
COMMENT ON COLUMN bronze_empresas.`ICAO` IS 'Designador de 3 letras fornecido pela Organização de Aviação Civil Internacional (OACI) às empresas aéreas, conforme DOC 8585';
COMMENT ON COLUMN bronze_empresas.`Razao` IS 'Nome de registro da empresa constante em seu ato constitutivo (Razão Social)';
COMMENT ON COLUMN bronze_empresas.`Servico` IS 'Atividades desempenhadas por empresas aéreas mediante remuneração, abrangendo transporte aéreo regular, não-regular e serviços aéreos especializados';
COMMENT ON COLUMN bronze_empresas.`Endereco` IS 'Endereço da sede que consta no Contrato ou Estatuto Social da empresa aérea';
COMMENT ON COLUMN bronze_empresas.`Cidade` IS 'Cidade em que a empresa aérea está sediada';
COMMENT ON COLUMN bronze_empresas.`UF` IS 'Unidade da federação em que a empresa aérea está sediada';
COMMENT ON COLUMN bronze_empresas.`CEP` IS 'CEP em que a empresa aérea está sediada';
COMMENT ON COLUMN bronze_empresas.`Telefone` IS 'Contato telefônico da empresa aérea';
COMMENT ON COLUMN bronze_empresas.`Email` IS 'Correio eletrônico da empresa aérea';
COMMENT ON COLUMN bronze_empresas.`Instrumento` IS 'Instrumento legal (Portaria) que autoriza, mediante autorização ou concessão, a exploração dos serviços aéreos públicos';
COMMENT ON COLUMN bronze_empresas.`Data` IS 'Data em que o instrumento legal (Decisão Operacional) foi publicado no Diário Oficial da União';
COMMENT ON COLUMN bronze_empresas.`Ativa` IS 'Indica se a empresa aérea está ativa ou inativa no cadastro da ANAC';
COMMENT ON COLUMN bronze_empresas.`Estrangeira` IS 'Indica se a empresa aérea é constituída em outro país (empresa estrangeira)';